# NB03 — Phase 3a: does the advantage survive rotation?

**Plan v2 §7.1. Budget: 4.0 h. Needs a single-GPU 80 GB pod.**

The key experiment.

**We plot three curves** 
We plot "R-id", "R-Q" and Qwen3-4B explainer curves so the claim becomes "R-Q falls to where
an unrelated 4B model sits".

**The explainer is initialized from unrotated `M`.** 
Because `M_Q ≡ M` as a function, feeding an unrotated-`M` explainer activations from `M_Q` holds behavioral similarity
identically fixed while destroying coordinate correspondence. We do not build an `M_Q` explainer.
This does not measure the self-explainer advantage, but rather the contribtion of the coordinate correspondence to that advantage.

| Hypothesis | Predicts for `E(from M)` reading `M_Q` |
|---|---|
| Behavioral self-simulation | advantage survives — behavior is bit-identical |
| Basis compatibility | advantage collapses toward the cross-model baseline |

**Arms** 
The core audit (NB00 sectoin 2) changed what this notebook has to run:

- `C0` under both rotations is the paper's own configuration. Each of the four `act_patch`
  configs do not build a projector. 
  They inject the target activation raw with a rank-128 LoRA downstream.
  This cannot represent `Q^{-1}` due to low rank, so we expect collapse due to capacity rather than self hood.
  We do this to test the exact set-up of the paper.
- `E_cross · P-rand` is the paper's cross-model condition. 
  We use a random dense projector with a rank-128 update (`C128`), because `model/utils.py:252–253` (on their repo) makes every trainable projector a LoRA target module. 
  A full-rank random projector is a different arm — `P-rand-full`, not to be conflated.
- `Cfull` remains the primary arm, and it is an augmentation we introduce so the question is
  answerable at all.

**We report everything normalized to the no-activation floor.** 
Rotation only destroys the coordinate frame information of `v`, but it has no effect on the information the explainer gains from the prompt itself. 
The paper's own `– activation` ablation bounds that value at ~4 points.
1 below measures our own floor at every `N` before any further runs.


In [ ]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


In [ ]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# Full sweep, three arms, LoRA + a full-rank input map on an 8B explainer.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="required", min_vram_gb=80)

import se_config as C


In [ ]:
import json
import math
import time

import pandas as pd
import torch

import se_common as S

tokenizer = S.load_tokenizer()
CROSS_MODEL_ID = "Qwen/Qwen3-4B"

datasets = {arm: S.load_ready_dataset(arm) for arm in ("identity", "Q")}
for arm, ds in datasets.items():
    print(f"{arm:>10}: {len(ds)} rows")

# matched data ordering across arms is a precondition, not a hope (v2 §3)
assert all(datasets["identity"][i]["messages"] == datasets["Q"][i]["messages"]
           for i in range(200)), "arms disagree on prompts — rebuild in NB02"
print("\narms share prompts, ordering, and labels; only the vectors differ")

# the explainer is loaded from the unrotated checkpoint — never from M_Q (v2 §2.1)
print(f"\nexplainer  : {C.EXPLAINER_MODEL_ID} (unrotated)")
print(f"cross model: {CROSS_MODEL_ID}")
print(f"N sweep    : {C.N_TRAIN_VALUES}")

# Every N runs every seed (supersedes v2 §3's "3 seeds at N in {512, 8192}"). Read out of
# C.seeds_for rather than restated, so this line cannot drift from the config the way it did
# when MULTI_SEED_N changed underneath it.
seed_grid = {n: C.seeds_for(n) for n in C.N_TRAIN_VALUES}
cut = [n for n, s in seed_grid.items() if len(s) < len(C.SEEDS)]
print(f"seeds      : {C.SEEDS} at every N — "
      f"{sum(len(s) for s in seed_grid.values())} (N x seed) cells per full-sweep arm")
if cut:
    print(f"             CUT to a single seed at N in {cut}; the seed band does not exist "
          f"there, and §9's 'overlap' vs 'separated' readings are undecidable at those N")
print(f"eval set   : {C.EVAL_SIZE} held-out examples "
      f"(8x the original; see NB00 §6 for why)")

# The paper's cross-model condition is a rank-capped projector, not a full-rank one (F.3).
# Reading these out of se_config keeps NB03, NB05 and NB07 from disagreeing about what
# "P-rand" means.
CROSS_CAP, CROSS_INIT = C.PROJECTOR_ARMS["P-rand"]
print(f"\nP-rand     = {CROSS_CAP} / init={CROSS_INIT}  "
      f"(rank {C.CAPACITY_RANK[CROSS_CAP]} update on a frozen random dense map)")
print(f"P-rand-full= {'/'.join(C.PROJECTOR_ARMS['P-rand-full'])}  (NB05 §5 runs the contrast)")


## 1. The no-activation floor — the scale every other number is on

Rotation will only destroy the signal from `v` and will not effect what the explainer gets from
the prompt itself. 
The paper bounds that value at 64.0 → 59.9 for Qwen3-8B self-explanation (Table 5).
So the entire measurable range of the rotation arm is only about 4 points, and we report all results normalized to this floor.

We need to rerun this to get our own floor because we are using different `N` and different eval-sets.
We set `v = 0` and the prompt, template, label distribution, the training budget all stay identical.


In [ ]:
FLOOR_N = C.N_TRAIN_VALUES     # cut to C.N_TRAIN_LADDER first if the budget bites

# v = 0 keeps prompts, labels, ordering and training budget identical: the only thing removed
# is the information carried by the activation.
zeroed = datasets["identity"].map(lambda ex: {
    "patch_position": {**ex["patch_position"],
                       "intervention_vector": [0.0] * len(
                           ex["patch_position"]["intervention_vector"])},
})

# Every seed, like every other arm — and here for one extra reason. The floor is the
# DENOMINATOR of `retained`, and `retained` is a ratio with a ~4-point denominator, so the
# floor's noise propagates into every normalized number more strongly than the numerator's
# does (prereg_threshold_justification.md §1.1: SE(retained) ~ 0.30 at retained = 0, rising
# with retained). Run single-seed, the denominator of every result in this project would be
# its least-replicated quantity.
floor_results = []
for n in FLOOR_N:
    for seed in C.seeds_for(n):
        t0 = time.time()
        print(f"\n=== no-activation floor (v = 0) · n={n} · seed={seed} " + "=" * 14)
        kw = dict(tokenizer=tokenizer, dataset=zeroed, seed=seed)
        S.run_training(n, "zerovec", "Cfull", "identity", **kw)
        scores = S.eval_run(n, "zerovec", "Cfull", "identity", **kw)
        scores["label"] = "no-activation floor (v=0)"
        floor_results.append(scores)
        print(f"  exact_match {scores['exact_match']:.3f} | "
              f"content_match {scores['content_match']:.3f} [{(time.time()-t0)/60:.1f} min]")

METRIC_COLS = ["exact_match", "has_changed_f1", "content_match"]

# One row per (N, seed) is kept so NB07 can put a band on the denominator; `floor_df` is the
# per-N mean, which is what everything downstream divides by.
floor_runs = pd.DataFrame(floor_results)
floor_runs.to_csv(f"{C.REPORTS_DIR}/no_activation_floor_runs.csv", index=False)

floor_df = floor_runs.groupby("n_train", as_index=False)[METRIC_COLS].mean()
floor_spread = (floor_runs.groupby("n_train", as_index=False)[METRIC_COLS]
                .agg(lambda x: x.max() - x.min()))
floor_df.to_csv(f"{C.REPORTS_DIR}/no_activation_floor.csv", index=False)
floor_spread.to_csv(f"{C.REPORTS_DIR}/no_activation_floor_spread.csv", index=False)
FLOOR = {int(r["n_train"]): r for _, r in floor_df.iterrows()}


def floor_at(n, metric="exact_match"):
    """The measured floor at N, falling back to the nearest measured N in log space."""
    n = int(n)
    if n in FLOOR:
        return FLOOR[n][metric]
    nearest = min(FLOOR, key=lambda k: abs(math.log2(k) - math.log2(n)))
    return FLOOR[nearest][metric]


untrained = S.eval_run(None, "identity", "Cfull", "identity", tokenizer,
                       dataset=datasets["identity"])

print("\nNO-ACTIVATION FLOOR (mean over seeds)")
print("=" * 62)
print(floor_df[["n_train"] + METRIC_COLS].round(3).to_string(index=False))
print("\nseed spread at each N (max - min) — this is the denominator's own uncertainty,")
print("and it enters every `retained` figure through the ratio:")
print(floor_spread[["n_train"] + METRIC_COLS].round(3).to_string(index=False))
print(f"\nuntrained explainer (no adapter at all): "
      f"exact_match {untrained['exact_match']:.3f}")
print(f"paper's Table 5 floor for comparison   : {C.PAPER_NO_ACTIVATION:.3f} "
      f"(their N, their eval set — not used in any reading)")
print("\nEvery arm below is reported twice: raw, and as the fraction of the gap between this")
print("floor and the unrotated reference that it retains. The second is the readable one.")


## 2. The sweep

We have a five arm sweeo.
The three full-sweep arms are the ones the figure turns on while `C0` runs at `N_TRAIN_LADDER` only.

| Arm | Whose | What it is |
|---|---|---|
| `E_self · Cfull · R-id` | ours | the augmented self-explainer, unrotated |
| `E_self · Cfull · R-Q` | ours | the same, reading a rotated frame |
| `E_cross · P-rand` | **the paper's** | random dense projector + rank-128 update (F.3) |
| `E_self · C0 · R-id` | **the paper's** | no map at all — the act_patch configuration (F.1) |
| `E_self · C0 · R-Q` | **the paper's** | the same, rotated: cannot represent `Q^{-1}` |

Each cell writes its own directory and is skipped if complete, so this is resumable across pod
restarts. It is ordered by `N` and then by arm, so if the pod dies halfway we have every arm at small `N` rather
than one arm everywhere.

Three full-sweep arms across seven `N` at three seeds (63), plus `C0` at two `N` under two rotations at three seeds (12), plus §1's floor at seven `N` at three seeds (21), plus §3's init control (4).



In [ ]:
ARMS = [
    # (label,                  rotation,   capacity,  init,       explainer,            N values)
    ("E_self · Cfull · R-id", "identity", "Cfull",   "identity", C.EXPLAINER_MODEL_ID, C.N_TRAIN_VALUES),
    ("E_self · Cfull · R-Q",  "Q",        "Cfull",   "identity", C.EXPLAINER_MODEL_ID, C.N_TRAIN_VALUES),
    ("E_cross · P-rand",      "identity", CROSS_CAP, CROSS_INIT, CROSS_MODEL_ID,       C.N_TRAIN_VALUES),
    # the paper's own configuration, under both rotations (revised §7.1)
    ("E_self · C0 · R-id",    "identity", "C0",      "identity", C.EXPLAINER_MODEL_ID, C.N_TRAIN_LADDER),
    ("E_self · C0 · R-Q",     "Q",        "C0",      "identity", C.EXPLAINER_MODEL_ID, C.N_TRAIN_LADDER),
]

results = []
for n in C.N_TRAIN_VALUES:
    for seed in C.seeds_for(n):
        for label, rot, cap, init, expl, n_values in ARMS:
            if n not in n_values:
                continue
            t0 = time.time()
            print(f"\n=== {label} · n={n} · seed={seed} " + "=" * 24)
            kw = dict(tokenizer=tokenizer, dataset=datasets[rot],
                      explainer_model_id=expl, seed=seed)
            S.run_training(n, rot, cap, init, **kw)
            scores = S.eval_run(n, rot, cap, init, **kw)
            scores["label"] = label
            results.append(scores)
            print(f"  exact_match {scores['exact_match']:.3f} | "
                  f"has_changed_f1 {scores['has_changed_f1']:.3f} | "
                  f"content_match {scores['content_match']:.3f} "
                  f"[{(time.time()-t0)/60:.1f} min]")

core = pd.DataFrame(results)
core.to_csv(f"{C.REPORTS_DIR}/core_sweep.csv", index=False)
core.groupby(["label", "n_train"]).exact_match.agg(["mean", "min", "max", "count"]).round(3)


## 3. The init-head-start control

`Cfull` under R-id is initialized at identity (at the solution) whereas `Cfull` under R-Q is initialized at identity when the truth is `Q^T`. 
So if R-id separates from R-Q at low `N` and converges at high `N` then this is a consequence of initialization and not self-explanation.

`Cfull-rand` starts both rotation arms at a random orthogonal map, putting them the same distance from their respective solutions.
Two `N` values is enough to bound it.

In [ ]:
rand_results = []
for n in C.N_TRAIN_LADDER:
    for rot in ("identity", "Q"):
        print(f"\n=== Cfull-rand · {C.ROTATION_LABEL[rot]} · n={n} " + "=" * 24)
        kw = dict(tokenizer=tokenizer, dataset=datasets[rot])
        S.run_training(n, rot, "Cfull-rand", "orthogonal", **kw)
        scores = S.eval_run(n, rot, "Cfull-rand", "orthogonal", **kw)
        scores["label"] = f"E_self · Cfull-rand · {C.ROTATION_LABEL[rot]}"
        rand_results.append(scores)
        print(f"  exact_match {scores['exact_match']:.3f}")

if rand_results:
    rd = pd.DataFrame(rand_results)
    print()
    print(rd.pivot_table(index="n_train", columns="rotation",
                         values="exact_match").round(3).to_string())
    print("\nA gap that survives here is not an initialization artifact.")


## 4. Normalized to the floor, which is where the readings live

`retained = (score − floor) / (score(Cfull, R-id) − floor)` at matched `N`:

- `1.0` — the arm keeps everything the activation was contributing
- `0.0` — the arm is at the no-activation floor; rotation destroyed all of it
- `< 0` — worse than passing no activation at all, which is a bug or an actively misleading
  vector, and §9 says investigate before reporting

`destroyed = 1 − retained` is what the preregistered thresholds are stated in (`overlap_destroyed = 0.15`, `separated_destroyed = 0.35`). 
The raw scale is printed alongside, because a reader comparing against the paper's tables needs it.

**We use a paired difference** 
Every arm is scored on the same held-out examples, so we need paired difference.
McNemar's discordant counts give `se = √(b + c) / n`, several times tighter than `√2 ×` the per-arm half-width.


In [ ]:
METRIC = "exact_match"

ref_by_n = (core[core.label == "E_self · Cfull · R-id"]
            .groupby("n_train")[METRIC].mean().to_dict())

rows = []
for _, r in core.iterrows():
    n = int(r["n_train"])
    fl, ref = floor_at(n, METRIC), ref_by_n.get(n)
    rows.append({
        "label": r["label"], "n_train": n, "seed": r["seed"],
        "raw": r[METRIC], "floor": fl, "reference": ref,
        "retained": S.fraction_retained(r[METRIC], fl, ref) if ref is not None else float("nan"),
    })
norm = pd.DataFrame(rows)
norm["destroyed"] = 1 - norm["retained"]
norm.to_csv(f"{C.REPORTS_DIR}/core_sweep_normalized.csv", index=False)

print(f"RETAINED FRACTION of the activation's contribution ({METRIC})")
print("=" * 78)
print(norm.pivot_table(index="n_train", columns="label", values="retained", aggfunc="mean")
      .round(3).to_string())
print("\nraw scores, same layout:")
print(norm.pivot_table(index="n_train", columns="label", values="raw", aggfunc="mean")
      .round(3).to_string())
print("\nfloor by N:")
print(floor_df.set_index("n_train")[METRIC].round(3).to_string())

# --- paired differences on the same eval items ------------------------------
print("\nPAIRED R-id vs R-Q at Cfull (same held-out examples, McNemar)")
print("=" * 78)
paired_rows = []
for n in C.N_TRAIN_VALUES:
    a = S.load_eval_records(n, "identity", "Cfull", "identity")
    b = S.load_eval_records(n, "Q", "Cfull", "identity")
    if not a or not b:
        continue
    d = S.paired_delta(a, b, key=METRIC)
    span = ref_by_n.get(n, 0) - floor_at(n, METRIC)
    d.update(n_train=n, destroyed=(d["delta"] / span) if span else float("nan"))
    paired_rows.append(d)
    print(f"  N={n:>5}: delta {d['delta']:+.4f} +/- {1.96 * d['se']:.4f}  "
          f"(discordant {d['discordant_a']}/{d['discordant_b']})  "
          f"= {d['destroyed']:+.2f} of the activation's contribution")

if paired_rows:
    with open(f"{C.REPORTS_DIR}/paired_deltas.json", "w") as f:
        json.dump(paired_rows, f, indent=2)
    print("\nA paired CI that excludes 0 is the strongest statement this eval size supports.")
else:
    print("  (no eval_records.json found — they are written by eval_run alongside the scores)")


## 5. The figure that answers the question

Raw scores on top with the floor too. 
Below, the fraction of the activations contribution.

The normalized panel is a row because the floor and the reference both move with `N`.


In [ ]:
import matplotlib.pyplot as plt

METRICS = ["exact_match", "has_changed_f1", "content_match"]
COLORS = {"E_self · Cfull · R-id": "#1b6ca8",
          "E_self · Cfull · R-Q": "#c0392b",
          "E_cross · P-rand": "#7f8c8d",
          "E_self · C0 · R-id": "#2e86c1",
          "E_self · C0 · R-Q": "#e67e22"}
DASHED = {"E_self · C0 · R-id", "E_self · C0 · R-Q"}      # the paper's configuration

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
for j, m in enumerate(METRICS):
    ax = axes[0][j]
    for label, color in COLORS.items():
        sub = core[core.label == label].groupby("n_train")[m].agg(["mean", "min", "max"])
        if not len(sub):
            continue
        sub = sub.sort_index()
        ax.plot(sub.index, sub["mean"], marker="o", color=color, label=label,
                linestyle="--" if label in DASHED else "-")
        ax.fill_between(sub.index, sub["min"], sub["max"], color=color, alpha=0.18)
    fl = floor_df.set_index("n_train")[m].sort_index()
    ax.plot(fl.index, fl.values, color="black", linestyle=":", linewidth=1.4,
            label="no-activation floor (v=0)")
    ax.set_xscale("log", base=2)
    ax.set_title(m.replace("_", " "))
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)

    # normalized row: 0 = the floor, 1 = the unrotated reference
    axn = axes[1][j]
    ref_m = core[core.label == "E_self · Cfull · R-id"].groupby("n_train")[m].mean().to_dict()
    for label, color in COLORS.items():
        sub = core[core.label == label]
        if not len(sub):
            continue
        g = sub.groupby("n_train")[m].mean().sort_index()
        y = [S.fraction_retained(v, floor_at(n, m), ref_m.get(int(n), float("nan")))
             for n, v in g.items()]
        axn.plot(g.index, y, marker="o", color=color,
                 linestyle="--" if label in DASHED else "-")
    axn.axhline(0, color="black", linestyle=":", linewidth=1.4)
    axn.axhline(1, color="#1b6ca8", linestyle="--", linewidth=0.8)
    axn.set_xscale("log", base=2)
    axn.set_xlabel("N_TRAIN")
    axn.set_ylim(-0.4, 1.3)
    axn.grid(alpha=0.3)

axes[0][0].set_ylabel("score (raw)")
axes[1][0].set_ylabel("fraction of activation\ncontribution retained")
axes[0][0].legend(fontsize=7.5)
fig.suptitle("Rotation vs the cross-model baseline, with the paper's own C0 configuration.\n"
             "Bands = min/max over seeds. Bottom row: 0 = no-activation floor, 1 = unrotated "
             "Cfull. The floor moves with N, which is why this is a second row rather than a "
             "second axis.")
fig.tight_layout()
fig.savefig(f"{C.FIGURES_DIR}/core_sweep.png", dpi=150)
plt.show()


In [ ]:
# the gap, the band, and the normalized scale the readings are stated on
prereg = json.load(open(f"{C.REPORTS_DIR}/preregistration.json"))
primary = prereg["primary_metric"]
TH = prereg["thresholds"]

piv = core.pivot_table(index="n_train", columns="label", values=primary, aggfunc="mean")
piv["gap (R-id - R-Q)"] = (piv.get("E_self · Cfull · R-id", 0)
                           - piv.get("E_self · Cfull · R-Q", 0))
piv["floor"] = [floor_at(n, primary) for n in piv.index]
piv["destroyed"] = [
    1 - S.fraction_retained(piv.loc[n].get("E_self · Cfull · R-Q", float("nan")),
                            floor_at(n, primary),
                            piv.loc[n].get("E_self · Cfull · R-id", float("nan")))
    for n in piv.index
]

spread = core.groupby(["label", "n_train"])[primary].agg(lambda x: x.max() - x.min())
band_width = spread[spread > 0]

print(f"{primary}: raw scores, the raw gap, and the normalized quantity\n")
print(piv.round(4).to_string())
print("\nseed band width (max - min), where >1 seed was run:")
print(band_width.round(4).to_string() if len(band_width) else "  (single seed everywhere)")
print(f"\nThresholds are on the NORMALIZED scale (revised §9): overlap if destroyed < "
      f"{TH['overlap_destroyed']}, separated if destroyed > {TH['separated_destroyed']} "
      f"and the seed bands do not overlap.")
print(f"Raw-scale equivalents at the largest N: overlap < "
      f"{TH['overlap_destroyed'] * (piv['E_self · Cfull · R-id'].iloc[-1] - piv['floor'].iloc[-1]):.4f}, "
      f"separated > "
      f"{TH['separated_destroyed'] * (piv['E_self · Cfull · R-id'].iloc[-1] - piv['floor'].iloc[-1]):.4f} "
      f"exact-match points — which is why the raw thresholds from v1 of the preregistration "
      f"were unreachable.")


**Do not classify the outcome here.** NB07 applies the preregistered rules mechanically to
everything at once. Reading this table and deciding what it means is the step preregistration
exists to prevent.

**Exit criteria (revised §7.1).** One figure: `Cfull` under both rotations, `C0` under both, the
cross-model baseline at each `N`, seed bands throughout, and the second row showing the fraction
of the activation's contribution retained. If `C0` collapses and `Cfull` does not, that is a
capacity result about the paper's configuration — label it, and do not let it stand in for the
basis finding.

Next: **NB04** turns capacity from a pinned constant into the measurement, and runs the exactness
check that catches plumbing bugs for free.
